In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("D:\\Data science\\Assignments\\Questions\\Recommendation System\\Recommendation System\\anime.csv")
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [14]:
display(df.isnull().sum())

df.dropna(inplace=True)

display(df.isnull().sum())

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [15]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify categorical and numerical features
categorical_features = ['type']
numerical_features = ['members', 'rating']

# Create transformers for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a column transformer to apply different transformers to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)])

# Create a pipeline to apply the preprocessing steps
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Apply the preprocessing to the dataframe
df_processed = pipeline.fit_transform(df)

# Convert the processed data back to a DataFrame (optional, for inspection)
# Get the new column names after one-hot encoding
new_column_names = numerical_features + list(pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features))
df_processed = pd.DataFrame(df_processed, columns=new_column_names)

display(df_processed.head())

,members,rating,type_Movie,type_Music,type_ONA,type_OVA,type_Special,type_TV,type_Unknown
0,3.330241,2.246515,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,14.148406,2.164612,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.754713,2.157167,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,11.957666,2.097601,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,2.429742,2.090156,0.0,0.0,0.0,0.0,0.0,1.0,0.0



Recommends anime based on cosine similarity with a threshold.

Args:  
    anime_name (str): The name of the target anime.  
    df (pd.DataFrame): The original dataframe with anime names.  
    df_processed (pd.DataFrame): The preprocessed dataframe with numerical features.  
    n_recommendations (int): The maximum number of recommendations to return.  
    similarity_threshold (float): The minimum cosine similarity score for a recommendation.

Returns:  
    list: A list of recommended anime names.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_anime(anime_name, df, df_processed, n_recommendations=10, similarity_threshold=0.8):
    # Get the index of the target anime
    try:
        anime_index = df[df['name'] == anime_name].index[0]
    except IndexError:
        return "Anime not found in the dataset."

    # Get the preprocessed features for the target anime
    anime_features = df_processed.iloc[anime_index].values.reshape(1, -1)

    # Calculate cosine similarity between the target anime and all other anime
    cosine_sim = cosine_similarity(anime_features, df_processed)

    # Get the indices and scores of the most similar anime
    sim_scores = list(enumerate(cosine_sim[0]))

    # Sort the anime by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Exclude the target anime itself from the recommendations and apply the threshold
    sim_scores = [score for i, score in sim_scores if i != anime_index and score >= similarity_threshold]

    # Get the top n_recommendations based on the threshold
    top_n_scores = sim_scores[:n_recommendations]

    # Get the indices of the top recommendations
    top_n_indices = [i for i, score in enumerate(cosine_sim[0]) if score >= similarity_threshold and i != anime_index][:n_recommendations]


    # Get the names of the recommended anime
    recommended_anime = df.iloc[top_n_indices]['name'].tolist()

    return recommended_anime

# Example usage:
# recommendations = recommend_anime('Your Anime Name Here', df, df_processed, similarity_threshold=0.9)
# print(recommendations)

In [19]:
anime_name = input()
recommendations = recommend_anime(anime_name, df, df_processed, similarity_threshold=0.9)
print(recommendations)

['Steins;Gate', 'Hunter x Hunter (2011)', 'Clannad: After Story', 'Gintama', 'Code Geass: Hangyaku no Lelouch R2', 'Sen to Chihiro no Kamikakushi', 'Shigatsu wa Kimi no Uso', 'Ookami Kodomo no Ame to Yuki', 'Code Geass: Hangyaku no Lelouch', 'Cowboy Bebop']


In [22]:
# Example usage with different similarity thresholds
anime_to_recommend = 'Fullmetal Alchemist: Brotherhood'

print(f"Recommendations for '{anime_to_recommend}' with similarity_threshold = 0.8:")
recommendations_08 = recommend_anime(anime_to_recommend, df, df_processed, similarity_threshold=0.8)
print(recommendations_08)
print("-" * 30)

print(f"Recommendations for '{anime_to_recommend}' with similarity_threshold = 0.9:")
recommendations_09 = recommend_anime(anime_to_recommend, df, df_processed, similarity_threshold=0.9)
print(recommendations_09)
print("-" * 30)

print(f"Recommendations for '{anime_to_recommend}' with similarity_threshold = 0.95:")
recommendations_095 = recommend_anime(anime_to_recommend, df, df_processed, similarity_threshold=0.95)
print(recommendations_095)

Recommendations for 'Fullmetal Alchemist: Brotherhood' with similarity_threshold = 0.8:
['Kimi no Na wa.', 'Steins;Gate', 'Gintama&#039;', 'Hunter x Hunter (2011)', 'Clannad: After Story', 'Gintama', 'Code Geass: Hangyaku no Lelouch R2', 'Haikyuu!! Second Season', 'Sen to Chihiro no Kamikakushi', 'Shigatsu wa Kimi no Uso']
------------------------------
Recommendations for 'Fullmetal Alchemist: Brotherhood' with similarity_threshold = 0.9:
['Steins;Gate', 'Hunter x Hunter (2011)', 'Clannad: After Story', 'Gintama', 'Code Geass: Hangyaku no Lelouch R2', 'Sen to Chihiro no Kamikakushi', 'Shigatsu wa Kimi no Uso', 'Ookami Kodomo no Ame to Yuki', 'Code Geass: Hangyaku no Lelouch', 'Cowboy Bebop']
------------------------------
Recommendations for 'Fullmetal Alchemist: Brotherhood' with similarity_threshold = 0.95:
['Steins;Gate', 'Hunter x Hunter (2011)', 'Clannad: After Story', 'Gintama', 'Code Geass: Hangyaku no Lelouch R2', 'Sen to Chihiro no Kamikakushi', 'Shigatsu wa Kimi no Uso', 'Co

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# function to get 'relevant' anime based on a high similarity threshold for testing.
def get_relevant_anime(anime_name, df, df_processed, relevance_threshold=0.95):
    
    try:
        anime_index = df[df['name'] == anime_name].index[0]
    except IndexError:
        return []

    anime_features = df_processed.iloc[anime_index].values.reshape(1, -1)
    cosine_sim = cosine_similarity(anime_features, df_processed)
    sim_scores = list(enumerate(cosine_sim[0]))

    # Exclude the target anime and get relevant anime based on the relevance_threshold
    relevant_indices = [i for i, score in sim_scores if i != anime_index and score >= relevance_threshold]
    relevant_anime = df.iloc[relevant_indices]['name'].tolist()
    return relevant_anime

# Split the dataset for evaluation
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Evaluate the recommendation system on the test set
precision_scores = []
recall_scores = []
f1_scores = []

# Use a reasonable threshold for generating recommendations for evaluation
recommendation_threshold = 0.9

for index, row in df_test.iterrows():
    anime_name = row['name']
    recommended_list = recommend_anime(anime_name, df, df_processed, similarity_threshold=recommendation_threshold, n_recommendations=20) # Increased n_recommendations for better chance of overlap

    # Get 'relevant' anime for the current anime based on a higher relevance threshold
    relevant_list = get_relevant_anime(anime_name, df, df_processed)

    if relevant_list: # Only evaluate if there are relevant items
        # Calculate true positives, false positives, and false negatives
        true_positives = len(set(recommended_list) & set(relevant_list))
        false_positives = len(set(recommended_list) - set(relevant_list))
        false_negatives = len(set(relevant_list) - set(recommended_list))

        # Calculate precision, recall, and F1-score for the current anime
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)

# Calculate and print the average metrics
average_precision = np.mean(precision_scores) if precision_scores else 0
average_recall = np.mean(recall_scores) if recall_scores else 0
average_f1 = np.mean(f1_scores) if f1_scores else 0

print(f"Average Precision: {average_precision:.4f}")
print(f"Average Recall: {average_recall:.4f}")
print(f"Average F1-score: {average_f1:.4f}")

Average Precision: 0.1298
Average Recall: 0.0165
Average F1-score: 0.0241


## Interview questions

**What is Collaborative Filtering, and how does it work?**

Collaborative Filtering is a technique used by recommendation systems to predict user preferences or recommend items. It works by leveraging the collective intelligence of users or items. The core idea is that if multiple users have similar tastes or if items are similar in how users interact with them, then those similarities can be used to make recommendations.

There are generally two main types of collaborative filtering:

*   **User-Based Collaborative Filtering:** This approach recommends items to a user based on the preferences of other users who are similar to them. It works by finding users with similar historical interactions (e.g., ratings, purchases, views) and then recommending items that those similar users liked but the current user hasn't interacted with yet.

*   **Item-Based Collaborative Filtering:** This approach recommends items that are similar to the items a user has already liked or interacted with. It works by finding relationships between items based on how users interact with them. For example, if many users who watched movie A also watched movie B, then movie B might be recommended to a user who has watched movie A.

User-based filtering measures similarity between users, while item-based filtering measures similarity between items. The system then uses these similarity measures to generate recommendations.